In [ ]:
#!/usr/bin/env python3
"""
此Python脚本分别根据每个项目子文件夹中的三种不同类型的PNG图像(*0001.png, *0002.png, *0003.png)
生成三个独立的GIF动画。生成GIF时，会根据文件名中包含的时间信息（例如 time1, time2 等）对图片帧排序，
确保按照正确的时间顺序生成动画。GIF文件保存在对应的子文件夹中。
"""

import os
import re
import imageio.v2 as imageio

# 设置父目录路径（需与NCL代码中使用的路径一致）
parent_dir = "/home/weiji/restart_exam/20250221/PV_NCL"

# 每帧持续时间（秒）
frame_duration = 0.5  # 500ms per frame

# 定义要处理的文件后缀列表
suffixes = ["0001.png", "0002.png", "0003.png"]

# 定义一个函数，从文件名中提取时间信息（例如 "time1" 提取出数字 1）
def extract_time(fname):
    m = re.search(r'time(\d+)', fname)
    if m:
        return int(m.group(1))
    # 如果没有匹配到，采用文件名进行排序（防止报错）
    return fname

# 遍历父目录下的每个子文件夹
for subdir in os.listdir(parent_dir):
    subdir_path = os.path.join(parent_dir, subdir)
    if os.path.isdir(subdir_path):
        # 获取该子文件夹下的所有PNG文件
        all_png_files = [f for f in os.listdir(subdir_path) if f.endswith('.png')]
        if not all_png_files:
            continue

        # 针对每种后缀分别处理
        for suffix in suffixes:
            # 筛选出匹配当前后缀的文件
            png_files = [f for f in all_png_files if f.endswith(suffix)]
            if not png_files:
                continue

            # 根据文件名中的时间信息排序
            png_files.sort(key=extract_time)

            # 读取图片并累积帧
            images = []
            for fname in png_files:
                fpath = os.path.join(subdir_path, fname)
                try:
                    img = imageio.imread(fpath)
                    images.append(img)
                except Exception as e:
                    print(f"读取 {fpath} 时出错: {e}")
                    continue

            if images:
                # 定义输出GIF文件名，例如 composite_PV_{子文件夹名}_0001.gif
                gif_filename = os.path.join(subdir_path, f"composite_PV_{subdir}_{suffix.replace('.png', '')}.gif")
                imageio.mimsave(gif_filename, images, duration=frame_duration)
                print(f"生成GIF: {gif_filename}")
            else:
                print(f"在 {subdir_path} 中未找到匹配 {suffix} 的图片来生成GIF。")